# Biblioteca musical en Colab

Descarga audio de una URL o lista de reproducción, lo convierte a MP3 con su carátula y metadatos, arma una biblioteca en JSON y clasifica cada canción por "estado de ánimo" a partir del título, artista y álbum.

> Úsalo solo con música que tengas derecho a descargar (contenido propio, libre o con permiso del autor).

Ejecuta las celdas en orden.

## 1. Instalar dependencias

In [ ]:
!pip install -q -U yt-dlp mutagen
!apt-get -y -qq install ffmpeg

## 2. Configuración

In [ ]:
import json
from pathlib import Path

from mutagen import File
from yt_dlp import YoutubeDL

BASE_DIR = Path("/content/music")   # carpeta de las canciones
DB_FILE = Path("/content/library.json")  # biblioteca en JSON
BASE_DIR.mkdir(parents=True, exist_ok=True)

## 3. Función de descarga

Guarda cada canción en `music/<lista o "Sueltas">/<n.º> - <título>.mp3`.

In [ ]:
def download_audio(url, folder=BASE_DIR):
    opts = {
        "format": "bestaudio/best",
        "outtmpl": str(folder / "%(playlist_title|Sueltas)s/%(playlist_index|00)s - %(title)s.%(ext)s"),
        "ignoreerrors": True,
        "noplaylist": False,
        "writethumbnail": True,
        "postprocessors": [
            {"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"},
            {"key": "FFmpegMetadata"},
            {"key": "EmbedThumbnail"},
        ],
    }
    with YoutubeDL(opts) as ydl:
        return ydl.extract_info(url, download=True)

## 4. Descargar

In [ ]:
url = input("Pega la URL o la lista de reproducción: ")
download_audio(url)
print("Descarga terminada")

## 5. Escanear la música descargada

In [ ]:
def scan_music(folder=BASE_DIR):
    songs = []
    for file in sorted(folder.rglob("*.mp3")):
        try:
            audio = File(file, easy=True)
        except Exception:  # archivo dañado: se usa el nombre del archivo
            audio = None
        tag = lambda key, default: (audio.get(key, [default])[0] if audio else default)
        songs.append({
            "title": tag("title", file.stem),
            "artist": tag("artist", "Desconocido"),
            "album": tag("album", "Desconocido"),
            "path": str(file),
            "mood": None,
        })
    return songs

library = scan_music()
print(f"{len(library)} canciones encontradas")
library[:5]

## 6. Clasificar por estado de ánimo

Clasificación básica por palabras clave. Si ninguna coincide, queda como `desconocido`.

In [ ]:
MOOD_WORDS = {
    "amor": ["love", "amor", "contigo", "beso", "baby", "corazón", "romance"],
    "tristeza": ["sad", "triste", "llorar", "adiós", "alone", "solo", "perdí"],
    "fiesta": ["party", "dance", "baila", "perreo", "noche", "club"],
    "energia": ["gym", "power", "fuego", "duro", "rápido", "speed"],
    "calma": ["sleep", "relax", "calma", "piano", "lofi", "study"],
    "ira": ["hate", "odio", "rabia", "war", "enemy"],
}

def classify_mood(song):
    text = f"{song['title']} {song['artist']} {song['album']}".lower()
    scores = {mood: sum(word in text for word in words) for mood, words in MOOD_WORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "desconocido"

for song in library:
    song["mood"] = classify_mood(song)

def songs_by_mood(mood):
    return [s for s in library if s["mood"] == mood]

songs_by_mood("amor")

## 7. Guardar la biblioteca

In [ ]:
DB_FILE.write_text(json.dumps(library, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Biblioteca guardada en {DB_FILE}")

## 8. Descargar todo a tu computadora (opcional)

In [ ]:
!cd /content && zip -qr music.zip music library.json
from google.colab import files
files.download("/content/music.zip")